# Experiment 6: lower learning rate

The last tuning experiment before feature work. Experiments 2 to 5 established that
1000 trees at the default learning rate of 0.1 gets CV to 0.962141. The standard next
move is a smaller learning rate with more trees to compensate.

## Why this is one variable and not two

Learning rate and tree count are not independent. Halving the learning rate roughly
doubles the trees needed to reach the same fit, so sweeping learning rate at a fixed
tree count would confound "is a smaller step better" with "did this configuration run
out of trees".

So the variable under test is the learning rate, and the tree count follows it by a
fixed rule: `n_estimators = round(100 / learning_rate)`, holding the product constant
at 100. That is one decision with a deterministic consequence, not two free choices.
Everything else stays at anchor settings, seed 42, same folds.

The 0.1 level reproduces experiment 4 and is the determinism check for this notebook.

## What would make this a dead end

If the gain from 0.1 to 0.03 is smaller than the fold spread, learning rate is not
where the remaining 0.008 AUC lives, and the answer is to stop tuning and go to the
missingness question. That is the expected outcome and it is worth 20 minutes to
establish rather than assume.

In [ ]:
import csv
import time
from datetime import datetime, timezone
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

SEED = 42
N_SPLITS = 5
TARGET = "addicted_label"
ID = "id"
CAT_COLS = ["gender", "stress_level", "academic_work_impact"]

LR_GRID = [0.1, 0.05, 0.03]
BUDGET = 100  # n_estimators = BUDGET / lr
REF_CV = 0.962141  # experiment 4: lr 0.1, 1000 trees

print("lightgbm", lgb.__version__, "| pandas", pd.__version__, "| numpy", np.__version__)
for lr in LR_GRID:
    print(f"  lr={lr}  ->  n_estimators={round(BUDGET / lr)}")

In [ ]:
def locate():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "data" / "raw" / "train.csv").exists():
            return base, base / "data" / "raw"
    kag = Path("/kaggle/input/playground-series-s6e8")
    if (kag / "train.csv").exists():
        return Path("/kaggle/working"), kag
    raise FileNotFoundError("could not find train.csv locally or under /kaggle/input")


REPO, RAW = locate()
SUB_DIR = REPO / "submissions"
OOF_DIR = REPO / "artifacts" / "oof"
SUB_DIR.mkdir(parents=True, exist_ok=True)
OOF_DIR.mkdir(parents=True, exist_ok=True)

train = pd.read_csv(RAW / "train.csv")
test = pd.read_csv(RAW / "test.csv")
sample = pd.read_csv(RAW / "sample_submission.csv")

FEATURES = [c for c in train.columns if c not in (ID, TARGET)]
for c in CAT_COLS:
    levels = pd.Categorical(pd.concat([train[c], test[c]], ignore_index=True)).categories
    train[c] = pd.Categorical(train[c], categories=levels)
    test[c] = pd.Categorical(test[c], categories=levels)

y = train[TARGET].to_numpy()

assert ID not in FEATURES, "id must never be a feature"
assert TARGET not in FEATURES, "target must never be a feature"
assert not (set(train[ID]) & set(test[ID])), "train and test ids overlap"
assert list(FEATURES) == [c for c in test.columns if c != ID], "train/test feature mismatch"

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
folds = np.full(len(train), -1, dtype=int)
for i, (_, va) in enumerate(skf.split(train, y)):
    folds[va] = i
assert (folds >= 0).all(), "every row must land in exactly one fold"

print(f"{len(train):,} train rows, {len(FEATURES)} features, {N_SPLITS} folds")

In [ ]:
def run_level(lr, n_estimators):
    oof = np.zeros(len(train), dtype=float)
    test_pred = np.zeros(len(test), dtype=float)
    fold_scores = []
    t0 = time.time()

    for f in range(N_SPLITS):
        tr_m, va_m = folds != f, folds == f
        model = lgb.LGBMClassifier(n_estimators=n_estimators, learning_rate=lr,
                                   random_state=SEED, verbose=-1)
        model.fit(train.loc[tr_m, FEATURES], y[tr_m])
        p_va = model.predict_proba(train.loc[va_m, FEATURES])[:, 1]
        oof[va_m] = p_va
        test_pred += model.predict_proba(test[FEATURES])[:, 1] / N_SPLITS
        fold_scores.append(roc_auc_score(y[va_m], p_va))

    return {"lr": lr, "n_estimators": n_estimators,
            "cv_mean": float(np.mean(fold_scores)), "cv_std": float(np.std(fold_scores)),
            "pooled": float(roc_auc_score(y, oof)), "secs": time.time() - t0,
            "oof": oof, "test_pred": test_pred}


results = []
for lr in LR_GRID:
    r = run_level(lr, round(BUDGET / lr))
    results.append(r)
    print(f"  lr={lr:<5} n={r['n_estimators']:>5}  cv={r['cv_mean']:.6f} "
          f"+/- {r['cv_std']:.6f}  pooled={r['pooled']:.6f}  {r['secs']:.0f}s")

## Determinism check

In [ ]:
repro = next(r for r in results if r["lr"] == 0.1)
delta = abs(repro["cv_mean"] - REF_CV)
print(f"experiment 4 CV : {REF_CV:.6f}")
print(f"reproduced CV   : {repro['cv_mean']:.6f}")
print(f"delta           : {delta:.9f}")
assert delta < 1e-6, (
    f"pipeline is not deterministic: lr 0.1 gave {repro['cv_mean']:.6f}, "
    f"experiment 4 gave {REF_CV:.6f}. Stop and diagnose."
)
print("\ndeterministic")

## Read it

The bar is the fold spread, not zero. A gain smaller than the spread is not a gain.

In [ ]:
best = max(results, key=lambda r: r["cv_mean"])
print(f"{'lr':>6} {'n_est':>7} {'cv':>10} {'sd':>9} {'vs exp4':>10} {'vs sd':>7} {'secs':>6}")
print("-" * 60)
for r in results:
    gain = r["cv_mean"] - REF_CV
    in_sd = gain / r["cv_std"] if r["cv_std"] else float("nan")
    print(f"{r['lr']:>6} {r['n_estimators']:>7} {r['cv_mean']:>10.6f} {r['cv_std']:>9.6f} "
          f"{gain:>+10.6f} {in_sd:>7.1f} {r['secs']:>6.0f}")

gain = best["cv_mean"] - REF_CV
print(f"\nbest: lr={best['lr']} at {best['cv_mean']:.6f}, {gain:+.6f} vs experiment 4")
if gain < best["cv_std"]:
    print("\nVERDICT: inside one fold standard deviation. Learning rate is not where "
          "the remaining gap lives. Stop tuning and go to the missingness question.")
else:
    print(f"\nVERDICT: {gain / best['cv_std']:.1f} fold sd. Real, but confirm across "
          "seeds before it goes in a final blend.")
if best["lr"] == LR_GRID[-1]:
    print("Best is at the edge of the grid, so the curve has not turned over.")

In [ ]:
LEDGER = REPO / "experiments.csv"
COLUMNS = ["id", "utc", "name", "cv_mean", "cv_std", "folds",
           "lb_public", "lb_private", "submitted", "notes"]

rows = []
if LEDGER.exists():
    with LEDGER.open(newline="", encoding="utf-8") as fh:
        rows = list(csv.DictReader(fh))
next_id = max((int(r["id"]) for r in rows), default=0) + 1

for r in results:
    tag = f"lgbm_lr{str(r['lr']).replace('.', '')}_n{r['n_estimators']}_seed{SEED}"
    np.save(OOF_DIR / f"{tag}.npy", r["oof"])
    sub = sample.copy()
    sub[TARGET] = r["test_pred"]
    sub.to_csv(SUB_DIR / f"{tag}.csv", index=False)

    note = (f"lr={r['lr']}, n_estimators={r['n_estimators']} scaled to hold lr*n=100, "
            f"all else anchor")
    if r["lr"] == 0.1:
        note = "reproducibility re-run of exp 4, same config, not a new idea"

    rows.append({
        "id": str(next_id), "utc": datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M"),
        "name": f"lgbm_lr{str(r['lr']).replace('.', '')}", "cv_mean": f"{r['cv_mean']:.6f}",
        "cv_std": f"{r['cv_std']:.6f}", "folds": str(N_SPLITS),
        "lb_public": "", "lb_private": "", "submitted": "no", "notes": note,
    })
    next_id += 1

with LEDGER.open("w", newline="", encoding="utf-8") as fh:
    w = csv.DictWriter(fh, fieldnames=COLUMNS)
    w.writeheader()
    w.writerows({c: r.get(c, "") for c in COLUMNS} for r in rows)

pd.read_csv(LEDGER)